Dependências
- Instala o kagglehub, caso necessário
- Carrega bibliotecas necessárias para execução

In [36]:
!pip install kagglehub[pandas-datasets]

In [48]:
# Dataset
import kagglehub
from kagglehub import KaggleDatasetAdapter
from typing import Tuple
import torch

# Pre-Processamento
import os
from pathlib import Path
from PIL import Image
from keras.preprocessing import image
import numpy as np
from keras.applications.imagenet_utils import preprocess_input
import random

# Train Test
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
import numpy as np
from sklearn.preprocessing import LabelEncoder

Carregar Dataset
- Primeiramente, baixa o dataset na maquina pelo kagglehub

In [38]:
file_path = "HAM10000_metadata.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "kmader/skin-cancer-mnist-ham10000",
  file_path,
)
root = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

print(df.head())

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
     lesion_id      image_id   dx dx_type   age   sex localization
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp
3  HAM_0002730  ISIC_0025661  bkl   histo  80.0  male        scalp
4  HAM_0001466  ISIC_0031633  bkl   histo  75.0  male          ear


In [52]:
labels = dict()

for i, l in  enumerate(df['dx'].unique()):
    labels[l] = i

In [53]:
class HAMDataset(Dataset):
    def __init__(self, dataframe, root, transform=None):
        # Fazer cópia e resetar índice
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.root = root
        self.transform = transform

        # Pré-processar: criar listas de caminhos e labels
        self.img_paths = []
        self.img_labels = []

        print("Pré-processando dataset...")
        for i in range(len(self.dataframe)):
            row = self.dataframe.iloc[i]
            img_name = row['image_id'] + '.jpg'
            diagnosis = row['dx']

            # Tentar part_1
            img_path = os.path.join(root, 'HAM10000_images_part_1', img_name)
            if not os.path.exists(img_path):
                img_path = os.path.join(root, 'HAM10000_images_part_2', img_name)

            if os.path.exists(img_path):
                self.img_paths.append(img_path)
                self.img_labels.append(labels[diagnosis])
            else:
                print(f"  Aviso: Imagem {img_name} não encontrada, pulando...")

        print(f"Dataset preparado: {len(self.img_paths)} imagens válidas de {len(dataframe)} totais")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        # idx já deve ser inteiro do DataLoader
        # Mas vamos garantir
        if torch.is_tensor(idx):
            idx = idx.item()

        # Acessar listas pré-processadas
        img_path = self.img_paths[idx]
        label = self.img_labels[idx]

        # Carregar imagem
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, label

In [54]:
# Combinação de transformações
vgg_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

dataset = HAMDataset(
    dataframe=df,
    root=root,
    transform=vgg_transform
)

dataloader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

Pré-processando dataset...
Dataset preparado: 10015 imagens válidas de 10015 totais


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# 1 Batch de exemplo
print(f"Tamanho dataset: {len(dataset)}")
print(f"Total batches: {len(dataloader)}")

for batch_idx, (images, labels) in enumerate(dataloader):
    print(f"\nBatch {batch_idx}:")
    print(f"  Images shape: {images.shape}")
    print(f"  Labels shape: {labels.shape}")
    print(f"  Labels: {labels}")

    print(f"  First image: min: {images[0].min():.3f}, max: {images[0].max():.3f}")

    if batch_idx == 0:
        break

In [57]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms


model = models.vgg16(pretrained=True)

# Congelar camadas convolucionais
for param in model.features.parameters():
    param.requires_grad = False

# Modificar classificador para 7 classes
num_classes = len(df['dx'].unique())
model.classifier[6] = nn.Linear(4096, num_classes)

# Mover para GPU se disponível
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# 4. Definir loss function e optimizer
criterion = nn.CrossEntropyLoss()
# Só otimizar os parâmetros que requerem gradiente
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

# 5. Função de treino para um epoch
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()  # Modo treino
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (images, labels) in enumerate(dataloader):
        # Mover para dispositivo
        images = images.to(device)
        labels = labels.to(device)

        # Zerar gradientes
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()
        optimizer.step()

        # Estatísticas
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        if batch_idx % 10 == 0:
            print(f'  Batch {batch_idx}/{len(dataloader)} - Loss: {loss.item():.4f}')

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

# 6. Treinar por alguns epochs
num_epochs = 3
print(f"Iniciando treino por {num_epochs} epochs...")

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    loss, acc = train_one_epoch(model, dataloader, criterion, optimizer, device)
    print(f"  Loss: {loss:.4f}, Accuracy: {acc:.2%}")

# 7. Testar com um batch
print("\nTestando modelo treinado...")
model.eval()  # Modo avaliação

with torch.no_grad():
    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        print(f"\nBatch de teste:")
        print(f"  Labels reais: {labels.cpu().numpy()}")
        print(f"  Predições:    {predicted.cpu().numpy()}")
        print(f"  Accuracy: {(predicted == labels).sum().item() / len(labels):.2%}")

        break

Iniciando treino por 3 epochs...

Epoch 1/3


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  Batch 0/313 - Loss: 1.9379
  Batch 10/313 - Loss: 1.3277
  Batch 20/313 - Loss: 1.2468
  Batch 30/313 - Loss: 1.4865
  Batch 40/313 - Loss: 1.0147
  Batch 50/313 - Loss: 0.8984
  Batch 60/313 - Loss: 0.8265
  Batch 70/313 - Loss: 0.9092
  Batch 80/313 - Loss: 1.2240
  Batch 90/313 - Loss: 0.5036
  Batch 100/313 - Loss: 0.6670
  Batch 110/313 - Loss: 1.2626
  Batch 120/313 - Loss: 0.9841
  Batch 130/313 - Loss: 1.1679
  Batch 140/313 - Loss: 0.9869
  Batch 150/313 - Loss: 0.8785
  Batch 160/313 - Loss: 1.0000
  Batch 170/313 - Loss: 1.7532
  Batch 180/313 - Loss: 1.0732
  Batch 190/313 - Loss: 1.0357
  Batch 200/313 - Loss: 1.3498
  Batch 210/313 - Loss: 0.6237
  Batch 220/313 - Loss: 1.0820
  Batch 230/313 - Loss: 0.9820
  Batch 240/313 - Loss: 1.2173
  Batch 250/313 - Loss: 1.0672
  Batch 260/313 - Loss: 1.0562
  Batch 270/313 - Loss: 1.1185
  Batch 280/313 - Loss: 0.8777
  Batch 290/313 - Loss: 1.1415
  Batch 300/313 - Loss: 1.4826
  Batch 310/313 - Loss: 0.7836
  Loss: 1.0493, Acc